## Import Libraries

In [1]:
import os
import sys
import copy
import time
import importlib
import numpy as np
import tensorflow as tf
import joblib as jl
import h5py
import pandas as pd

threads = 4

from tqdm import tqdm

In [2]:
# Plotting Set up

import matplotlib as mpl
from matplotlib import pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
%matplotlib notebook

# Get colors list
colors = mpl.rcParams['axes.prop_cycle'].by_key()['color']*10

# Font parameters
mpl.rcParams['text.latex.preamble'].join([
    r'\usepackage{amsmath}',
    r'\boldmath',
    r'\usepackage{pifont}',
    r'\usepackage{tikz}',
    r'\usepackage[utf8]{inputenc}'
])
mpl.rcParams['font.family'] = 'lmodern'

# Tick parameters
mpl.rcParams['xtick.labelsize'] = 40
mpl.rcParams['ytick.labelsize'] = 40
mpl.rcParams['xtick.major.size'] = 20
mpl.rcParams['xtick.major.width'] = 2
mpl.rcParams['xtick.minor.size'] = 10
mpl.rcParams['xtick.minor.width'] = 2

# Tick parameters
mpl.rcParams['axes.labelsize'] = 40
mpl.rcParams['axes.titlesize'] = 40
mpl.rcParams['legend.fontsize'] = 35


### Initialize SurQCT Parameters

In [3]:
from InputDataPostProcess import inputdata 

WORKSPACE_PATH      = os.environ['WORKSPACE_PATH']
InputData    = inputdata(WORKSPACE_PATH)

ExcitType           = 'KInel'
RunIdx            = 6
# ExcitType           = 'KExch'
# RunIdx            = 1

TTranVec            = [1500.0, 8000.0]

Molecules           = ['O2','O2']
Atoms               = ['O','O']
System = 'O3_UMN'

ZeroDRunIdx         = RunIdx

MinValueTrain       = 1.e-15

InputData.KineticFldr     = InputData.WORKSPACE_PATH+'/Air_Database/Run_0D_surQCT/database/kinetics/prode_'+System+'_Active_Run'+str(ZeroDRunIdx)+'/'
 

### Read Levels Data

In [4]:
from Reading  import read_levelsdata, read_diatdata

OtherVar           = InputData.OtherVar
xVarsVec_i         = InputData.xVarsVec_i 
xVarsVec_Delta     = InputData.xVarsVec_Delta
xVarsVec           = list(set(xVarsVec_i) | set(xVarsVec_Delta))

xbranch = [(VarName + '_i') for VarName in xVarsVec_i]
Tbranch = ['TTran_i']
xtrunk = [(VarName + '_Delta') for VarName in xVarsVec_Delta]
Ttrunk = ['TTran_Delta']
xTrainingVars = xbranch + Tbranch + xtrunk + Ttrunk

xVarsVec_i         = InputData.xVarsVec_i + ['vqn']
xVarsVec_Delta     = InputData.xVarsVec_Delta
xVarsVec           = list(set(xVarsVec_i) | set(xVarsVec_Delta))

NMolecules         = len(InputData.PathToDiatFile)
LevelsData = []
DiatData   = []
NLevels    = []

for iMol in range(NMolecules):
    
    LevelsDataTemp = read_levelsdata(InputData.PathToLevelsFile[iMol], xVarsVec, '')
    LevelsData.append(LevelsDataTemp)

    DiatDataTemp = read_diatdata(InputData.PathToDiatFile[iMol], InputData.Molecules[iMol], np.array(TTranVec), np.array(TTranVec))
    DiatData.append(DiatDataTemp)
    
    NLevelsTemp    = DiatDataTemp.shape[0]
    NLevels.append(NLevelsTemp)

[SurQCT]:   Reading Molecular Levels Data from: /home/venturi/WORKSPACE//Air_Database/Run_0D/database/levels/O2_UMN_log_nd.csv
[SurQCT]:   Reading Molecular Levels Data from: /home/venturi/WORKSPACE//CoarseAIR/coarseair/dtb/Molecules/O2/UMN/FromUMN_Sorted.inp
[SurQCT]:   Reading Molecular Levels Data from: /home/venturi/WORKSPACE//Air_Database/Run_0D/database/levels/O2_UMN_log_nd.csv
[SurQCT]:   Reading Molecular Levels Data from: /home/venturi/WORKSPACE//CoarseAIR/coarseair/dtb/Molecules/O2/UMN/FromUMN_Sorted.inp


### Initialize NN Model

In [5]:
prode_dir = os.environ['WORKSPACE_PATH'] + '/PRODE/prode/'

if(ExcitType == 'KInel'):   
    case_dir = "../run"+str(RunIdx)


if importlib.util.find_spec('prode') is None:
    sys.path.append(os.path.normpath(prode_dir))
sys.path.append(os.path.normpath(case_dir))
cwd = os.getcwd()

# PrODE -----------------------------------------------------------------------
from prode.utilities import utils
from prode.apps import functionObjects as fun
from prode.postprocessing import testing, training

# Extra modules ---------------------------------------------------------------
# Inputs
import inputs

os.chdir(os.path.normpath(case_dir))

net = fun._net(
    input_module=inputs
)

from inputs.general import flags

for k in ('train', 'save'):
    flags[k] = 0
if flags['load'] not in (1, 2):
    utils.raise_value_err(
        "The network has to be restored from a saved or a checkpointed "
        "one. Modified the 'load' entry in the `flags` input variable."
    )
model = fun._model(
    flags=flags,
    input_module=inputs,
    net=net
)


[PrODE]: Initializing 'OpNN' network ...

[PrODE]: Building 'OpNN' network ...
Model: "SurQCT_OpNN"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            [(None, 5)]          0                                            
__________________________________________________________________________________________________
input_2 (InputLayer)            [(None, 5)]          0                                            
__________________________________________________________________________________________________
branch_fn (SurQCTNormalization) (None, 5)            11          input_1[0][0]                    
__________________________________________________________________________________________________
trunk_fn (SurQCTNormalization)  (None, 5)            11          input_2[0][0]                    
________

### Writing Rates in Plato Format

In [6]:
def write_predictiondata(ExcitType, KineticFile, csvkinetics, iFlg, Molecules, Atoms, iLevel, jLevelVec, KVec):

    if (iFlg == 0):
        for i in range(len(jLevelVec)): 
            ProcName = Molecules[0] + '(' + str(iLevel+1) + ')+' + Atoms[0] + '=' + Molecules[1] + '(' + str(jLevelVec[i]+1) + ')+' + Atoms[1]
            if (KVec[i] >= MinValueTrain):
                if (ExcitType == 'KInel'):
                    Line     = ProcName + ':%.4e,+0.0000E+00,+0.0000E+00,5\n' % KVec[i]
                else:
                    Line     = ProcName + ':%.4e,+0.0000E+00,+0.0000E+00,6\n' % KVec[i]
                csvkinetics.write(Line)

    elif (iFlg == -1):
        print('[SurQCT]:   Writing Kinetics in File: ' + KineticFile )
        csvkinetics  = open(KineticFile, 'w')

    elif (iFlg == -2):
        print('[SurQCT]:   Closing Kinetics File: ' + KineticFile )
        csvkinetics.close()


    return csvkinetics

In [ ]:
try:
    os.makedirs(InputData.KineticFldr)
except OSError as e:
    pass

for TTran in TTranVec:
    print('[SurQCT]:    TTran = ', TTran)


    ### Opening Files
    PathToKineticFldr = InputData.KineticFldr + '/T' + str(int(TTran)) + 'K'
    try:
        os.makedirs(PathToKineticFldr)
    except OSError as e:
        pass

    if(ExcitType == 'KInel'):   
        KineticFile_KInel = PathToKineticFldr + '/Inel.dat'
        csvkinetics_KInel = write_predictiondata(ExcitType, KineticFile_KInel, None, -1, Molecules, Atoms, None, None, None)
    elif(ExcitType == 'KExch'):   
        KineticFile_KExch = PathToKineticFldr + '/Exch_Type1.dat'
        csvkinetics_KExch = write_predictiondata(ExcitType, KineticFile_KExch, None, -1, Molecules, Atoms, None, None, None)



    ### Loop on Initial States
    Str = 'q_'+str(int(TTran))
    for iIdx in tqdm(range(NLevels[0]), desc='[SurQCT]:     Generating Inelastic and Exchange Rate Matrixes'):
        time.sleep(0.001)
        
        if (InputData.ExoEndoFlg):
            jIdxVec           = [jIdx for jIdx in np.arange(NLevels[1]) if (DiatData[1]['EInt'].to_numpy()[jIdx] < DiatData[0]['EInt'].to_numpy()[iIdx])]
        else:
            jIdxVec           = [jIdx for jIdx in np.arange(NLevels[1]) if (DiatData[1][Str].to_numpy()[jIdx]    > DiatData[0][Str].to_numpy()[iIdx])]
        jNLevels              = len(jIdxVec)
        
        ### FWD Rates
        iiIdxVec              = [iIdx]*jNLevels
        
        TTranVec              = np.ones((jNLevels))*TTran
        TTranDataTemp         = pd.DataFrame({'TTran': TTranVec})
        TTranDataTemp.index   = jIdxVec

        
        iLevelsDataTemp       = LevelsData[0].iloc[iiIdxVec,:].copy()
        iLevelsDataTemp.index = jIdxVec

        jLevelsDataTemp       = LevelsData[1].iloc[jIdxVec,:].copy()        
        if (OtherVar == '_Delta'):
            jLevelsDataTemp   = iLevelsDataTemp.subtract(jLevelsDataTemp) 
        else:
            jLevelsDataTemp   = jLevelsDataTemp
        jLevelsDataTemp.index = jIdxVec
        
        kLevelsDataTemp       = LevelsData[1].iloc[jIdxVec,:].copy()        
        kLevelsDataTemp.index = jIdxVec

        
        iLevelsData           = pd.concat([iLevelsDataTemp[xVarsVec_i], TTranDataTemp], axis=1)
        iLevelsData.columns   = [(VarName + '_i') for VarName in iLevelsData.columns]
        
        jLevelsData           = pd.concat([jLevelsDataTemp[xVarsVec_Delta], TTranDataTemp], axis=1)
        jLevelsData.columns   = [(VarName + OtherVar) for VarName in jLevelsData.columns]

        kLevelsData           = pd.concat([kLevelsDataTemp[xVarsVec_i], TTranDataTemp], axis=1)
        kLevelsData.columns   = [(VarName + '_j') for VarName in kLevelsData.columns]
        
        xTemp_FWD             = pd.concat([iLevelsData, jLevelsData, kLevelsData], axis=1)

        x1 = iLevelsData[xTrainingVars[0:5]].to_numpy()
        x2 = jLevelsData[xTrainingVars[5:10]].to_numpy()    
        x = [x1,x2]
        
        if (len(xTemp_FWD[xTrainingVars]) > 0):
            if(ExcitType == 'KInel'): 
                KInel_NN_FWD = np.exp( model.net.predict(x) ) / InputData.MultFact  
                csvkinetics_KInel = write_predictiondata(ExcitType, KineticFile_KInel, csvkinetics_KInel, 0, Molecules, Atoms, iIdx, jIdxVec, KInel_NN_FWD)
            
            elif(ExcitType == 'KExch'):      
                KExch_NN_FWD = np.exp(  model.net.predict(x) ) / InputData.MultFact
                csvkinetics_KExch = write_predictiondata(ExcitType, KineticFile_KExch, csvkinetics_KExch, 0, Molecules, Atoms, iIdx, jIdxVec, KExch_NN_FWD)

                     
    if(ExcitType == 'KInel'): 
        csvkinetics_KInel = write_predictiondata(ExcitType, KineticFile_KInel, csvkinetics_KInel, -2, None, None, None, None, None)
    elif(ExcitType == 'KExch'):
        csvkinetics_KExch = write_predictiondata(ExcitType, KineticFile_KExch, csvkinetics_KExch, -2, None, None, None, None, None)



[SurQCT]:     Generating Inelastic and Exchange Rate Matrixes:   0%|          | 0/6115 [00:00<?, ?it/s]

[SurQCT]:    TTran =  1500.0
[SurQCT]:   Writing Kinetics in File: /home/venturi/WORKSPACE//Air_Database/Run_0D_surQCT/database/kinetics/prode_O3_UMN_Active_Run6//T1500K/Inel.dat


[SurQCT]:     Generating Inelastic and Exchange Rate Matrixes: 100%|██████████| 6115/6115 [17:55<00:00,  5.69it/s]
[SurQCT]:     Generating Inelastic and Exchange Rate Matrixes:   0%|          | 2/6115 [00:00<07:16, 14.01it/s]

[SurQCT]:   Closing Kinetics File: /home/venturi/WORKSPACE//Air_Database/Run_0D_surQCT/database/kinetics/prode_O3_UMN_Active_Run6//T1500K/Inel.dat
[SurQCT]:    TTran =  8000.0
[SurQCT]:   Writing Kinetics in File: /home/venturi/WORKSPACE//Air_Database/Run_0D_surQCT/database/kinetics/prode_O3_UMN_Active_Run6//T8000K/Inel.dat


[SurQCT]:     Generating Inelastic and Exchange Rate Matrixes:  49%|████▉     | 3013/6115 [07:30<15:39,  3.30it/s]